In [ ]:
%pip install transformers torch


In [ ]:
import torch
from transformers import DistilBertModel, DistilBertTokenizer

# Load the DistilBERT model and tokenizer
model = DistilBertModel.from_pretrained(
    "distilbert-base-uncased", 
    torch_dtype=torch.float16
)
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")


In [ ]:
# Tokenize the input text and prepare for the model
inputs = tokenizer(
    texts,
    padding=True,           # Pad sentences to the same length
    truncation=True,        # Truncate sentences to the model's max length
    return_tensors="pt"     # Return PyTorch tensors
)

In [ ]:
# Move inputs to the same device as the model (e.g., GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
inputs = {key: value.to(device) for key, value in inputs.items()}

# Get the embeddings from the last hidden state
with torch.no_grad():  # No need to compute gradients for inference
    outputs = model(**inputs)
    # `outputs.last_hidden_state` has shape (batch_size, seq_len, hidden_size)
    embeddings = outputs.last_hidden_state[:, 0, :]  # CLS token embeddings


In [ ]:
# Detach and move to CPU for further processing
embeddings = embeddings.detach().cpu()
# Normalize the embeddings to unit length
embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)


In [ ]:
import numpy as np

embeddings_np = embeddings.numpy()
